In [0]:
%run ../00.common/1.environment_config

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.drivers"
silver_table=f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df=spark.read.table(bronze_table)

In [0]:
drivers_dropped_df=drivers_df.drop("url")

In [0]:
drivers_renamed_df=(drivers_dropped_df
                    .withColumnsRenamed({"driverId":"driver_id","dateofbirth":"date_of_birth"})


)

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_concat_df=(drivers_renamed_df
                   .withColumn("driver_name",
                               F.initcap(F.concat_ws(" ","name.givenname","name.familyname")))
                   .drop(F.col("name"))
)

In [0]:
drivers_valid_df=(drivers_concat_df.dropDuplicates(['driver_id']))

In [0]:
drivers_final_df=(drivers_valid_df
                  .withColumn("nationality",F.initcap(F.col("nationality")))
)

In [0]:
(
    drivers_final_df
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))